In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockA (QRS_TOT) — 读取 QRS_TOT 数据 & 预处理 & 写缓存
#
# 功能：
#   1) 从 EXTR isobar 文件中，读取 QRS_TOT(time, lev, lat)
#      计算 60–90°N cosine-lat 加权平均：
#        QRS_pcap_60_90N(time, lev)
#   2) 从 merged h3 int 文件中，读取 QRS_TOT(time, plev, lat, lon)
#      先做 zonal mean，再做 60–90°N cosine-lat 加权平均：
#        QRS_pcap_60_90N(time, lev)
#   3) 将两套结果写入 NetCDF，供 BlockB/C/D/E/F 使用
#   4) O3 极端年信息从 General_O3 目录中读取（不再重新计算）
#
# 输入：
#   EXTR QRS climatology:
#     /mnt/backup_ETH/extr_2000/CO2x1SmidEmin_yBWCN/
#       CO2x1SmidEmin_yBWCN.cam.h1.0101-0300_T2Mz_QRS_TOT.isobar.nc
#     dims: time, lev(hPa), lat   (已是 zonal mean)
#
#   merged special-years source:
#     /mnt/backup_ETH/marina/waccm/CHEM_2000_restart/BWCN.e122.f19_g16.002/
#       BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc
#     dims: time, plev(Pa), lat, lon
#
# 输出：
#   QRS_TOT_pcap_EXTR_60_90N_lev_time.nc
#   QRS_TOT_pcap_MERGED_60_90N_lev_time.nc
# ============================================================

import os
import numpy as np
import xarray as xr

O3_ROOT  = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
QRS_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_QRS_TOT"
os.makedirs(QRS_ROOT, exist_ok=True)

EXTR_QRS_PATH = (
    "/mnt/backup_ETH/extr_2000/CO2x1SmidEmin_yBWCN/"
    "CO2x1SmidEmin_yBWCN.cam.h1.0101-0300_T2Mz_QRS_TOT.isobar.nc"
)
MERGED_QRS_PATH = (
    "/mnt/backup_ETH/marina/waccm/CHEM_2000_restart/BWCN.e122.f19_g16.002/"
    "BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc"
)

LOW10_TXT = os.path.join(O3_ROOT, "O3_lowest10_years.txt")
LOW25_TXT = os.path.join(O3_ROOT, "O3_lowest25pct_years.txt")

OUT_EXTR_NC   = os.path.join(QRS_ROOT, "QRS_TOT_pcap_EXTR_60_90N_lev_time.nc")
OUT_MERGED_NC = os.path.join(QRS_ROOT, "QRS_TOT_pcap_MERGED_60_90N_lev_time.nc")


def lev_to_pa(lev_vals):
    lev_vals = np.asarray(lev_vals)
    if np.nanmax(lev_vals) <= 2000.0:   # likely hPa
        return lev_vals * 100.0
    return lev_vals


def calc_pcap_weighted_mean(ds, var="QRS_TOT", lat_slice=(60, 90), do_lon_mean=False):
    """
    计算 60–90°N cosine-lat 加权平均 QRS_TOT(time, lev):

    - 若 do_lon_mean=True 且包含 lon → 先做 zonal mean
    - 取 lat=60–90N
    - weighted(cos(lat)).mean(lat)
    - 垂直维统一命名为 lev，单位统一 Pa
    """
    da = ds[var]

    if do_lon_mean and ("lon" in da.dims):
        da = da.mean(dim="lon")

    if "lat" not in da.dims:
        raise ValueError(f"{var} has no lat dim, dims={da.dims}")

    da_cap = da.sel(lat=slice(*lat_slice))

    w = np.cos(np.deg2rad(da_cap["lat"]))
    da_pcap = da_cap.weighted(w).mean(dim="lat")  # (time, vertical)

    # identify vertical dim
    lev_name = None
    for name in ("plev", "lev_p", "lev", "level"):
        if name in da_pcap.dims:
            lev_name = name
            break
    if lev_name is None:
        raise ValueError(f"{var} has no vertical dim in {da_pcap.dims}")

    lev_pa = lev_to_pa(da_pcap[lev_name].values)
    da_pcap = da_pcap.rename({lev_name: "lev"})
    da_pcap = da_pcap.assign_coords(lev=("lev", lev_pa))
    da_pcap["lev"].attrs["units"] = "Pa"

    da_pcap.name = f"{var}_pcap_60_90N"
    da_pcap.attrs["long_name"] = f"60–90N polar-cap cosine-lat weighted mean of {var}"
    da_pcap.attrs["units"] = da.attrs.get("units", "")

    return da_pcap


def main_blockA_QRS():
    # ---- sanity check: O3 extreme-year lists exist ----
    if not (os.path.exists(LOW10_TXT) and os.path.exists(LOW25_TXT)):
        raise FileNotFoundError(
            "O3 extreme-year txt files not found. Please run O3 BlockA first.\n"
            f"Expected: {LOW10_TXT} and {LOW25_TXT}"
        )
    lowest10_years = np.loadtxt(LOW10_TXT, dtype=int).reshape(-1)
    lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)
    print("[BlockA_QRS] Lowest-10 O3 years:", lowest10_years)
    print("[BlockA_QRS] Lowest-25% O3 years:", lowest25_years)

    # ---- 1) EXTR polar-cap mean ----
    print("[BlockA_QRS] Reading EXTR QRS_TOT:", EXTR_QRS_PATH)
    ds_extr = xr.open_dataset(EXTR_QRS_PATH)
    qrs_extr_pcap = calc_pcap_weighted_mean(ds_extr, var="QRS_TOT", do_lon_mean=False)
    ds_extr.close()

    years_extr = np.unique(qrs_extr_pcap.time.dt.year.values.astype(int))
    n_days = int(qrs_extr_pcap.time.dt.dayofyear.max())
    print("[BlockA_QRS] EXTR years:", years_extr[0], "→", years_extr[-1],
          "nyears=", len(years_extr), "n_days=", n_days)

    qrs_extr_pcap = qrs_extr_pcap.transpose("time", "lev")
    qrs_extr_pcap.to_netcdf(OUT_EXTR_NC)
    print("[BlockA_QRS] Saved EXTR pcap series to:", OUT_EXTR_NC)

    # ---- 2) MERGED polar-cap mean ----
    print("[BlockA_QRS] Reading merged QRS_TOT:", MERGED_QRS_PATH)
    ds_m = xr.open_dataset(MERGED_QRS_PATH)
    qrs_m_pcap = calc_pcap_weighted_mean(ds_m, var="QRS_TOT", do_lon_mean=True)
    ds_m.close()

    years_m = np.unique(qrs_m_pcap.time.dt.year.values.astype(int))
    n_days_m = int(qrs_m_pcap.time.dt.dayofyear.max())
    print("[BlockA_QRS] merged years:", years_m[0], "→", years_m[-1],
          "nyears=", len(years_m), "n_days=", n_days_m)

    qrs_m_pcap = qrs_m_pcap.transpose("time", "lev")
    qrs_m_pcap.to_netcdf(OUT_MERGED_NC)
    print("[BlockA_QRS] Saved merged pcap series to:", OUT_MERGED_NC)

    print("[BlockA_QRS] Done.")


if __name__ == "__main__":
    main_blockA_QRS()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockB (QRS_TOT) — 60–90°N QRS_TOT 折线图（10/50/100 hPa）
#
# 功能：
#   1) EXTR：10 个低 O3 年 vs 其它年份
#   2) merged：0008/0013/0014/0019 vs EXTR all / low25 气候态
#
#   时间轴：前一年 Oct–Dec (91 天) + 当年 Jan–Sep
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib import rcParams
from matplotlib.ticker import ScalarFormatter

O3_ROOT  = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
QRS_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_QRS_TOT"
os.makedirs(QRS_ROOT, exist_ok=True)

QRS_EXTR_NC   = os.path.join(QRS_ROOT, "QRS_TOT_pcap_EXTR_60_90N_lev_time.nc")
QRS_MERGED_NC = os.path.join(QRS_ROOT, "QRS_TOT_pcap_MERGED_60_90N_lev_time.nc")

LOW10_TXT = os.path.join(O3_ROOT, "O3_lowest10_years.txt")
LOW25_TXT = os.path.join(O3_ROOT, "O3_lowest25pct_years.txt")

N_PREV = 91
YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)


def get_level_index(lev_vals_pa, target_hpa):
    target_pa = target_hpa * 100.0
    idx = int(np.argmin(np.abs(lev_vals_pa - target_pa)))
    return idx, float(lev_vals_pa[idx] / 100.0)


def nice_upper_bound(vmax, n_ticks=5):
    """
    把 vmax round 到一个“干净”的上界，用于 ylim / ticks。
    """
    if not np.isfinite(vmax) or vmax <= 0:
        return 1.0
    exp = np.floor(np.log10(vmax))
    base = 10 ** exp
    frac = vmax / base
    # nice steps: 1,2,4,5,8,10
    if frac <= 1:
        nice = 1
    elif frac <= 2:
        nice = 2
    elif frac <= 4:
        nice = 4
    elif frac <= 5:
        nice = 5
    elif frac <= 8:
        nice = 8
    else:
        nice = 10
    return nice * base


def auto_set_yaxis(ax, series_list, level_label, debug_tag=""):
    """
    根据所有数据（极端年、其它年均值、包络）自动设置 y 轴范围和刻度，
    并强制使用 ScalarFormatter，避免长尾小数刻度。
    """
    all_vals = []
    for s in series_list:
        if s is None:
            continue
        arr = np.asarray(s).ravel()
        arr = arr[np.isfinite(arr)]
        if arr.size:
            all_vals.append(arr)

    if not all_vals:
        return

    all_vals = np.concatenate(all_vals)
    vmin = float(np.nanmin(all_vals))
    vmax = float(np.nanmax(all_vals))

    # 由于 QRS_TOT 基本为正，线图用 0 起更直观
    ymax = nice_upper_bound(max(vmax, 0.0))

    yticks = np.linspace(0.0, ymax, 5)

    ax.set_ylim(0.0, ymax)
    ax.set_yticks(yticks)

    fmt = ScalarFormatter(useMathText=True)
    fmt.set_scientific(True)
    fmt.set_powerlimits((-2, 2))
    ax.yaxis.set_major_formatter(fmt)

    # debug prints
    print(f"[YAXIS{debug_tag}] {level_label}: raw min={vmin:.3e}, raw max={vmax:.3e}")
    print(f"[YAXIS{debug_tag}] {level_label}: nice ymax={ymax:.3e}, yticks={yticks}")


def plot_extremes_vs_others(var_extr, lowest10_years, level_label, out_png, out_pdf):
    years_extr = np.unique(var_extr.time.dt.year.values.astype(int))
    n_days = int(var_extr.time.dt.dayofyear.max())
    print(f"[BlockB_QRS] EXTR years for {level_label}:", years_extr)
    print(f"[BlockB_QRS] Days per year (EXTR): {n_days}")

    fig, ax = plt.subplots(1, 1, figsize=(13, 10), constrained_layout=True)

    low_years = lowest10_years
    neutral_years = years_extr[~np.isin(years_extr, low_years)]

    var_low_cur = var_extr.sel(time=var_extr.time.dt.year.isin(low_years))
    var_low_prev = var_extr.sel(time=var_extr.time.dt.year.isin(low_years - 1))

    var_neu_cur = var_extr.sel(time=var_extr.time.dt.year.isin(neutral_years))
    var_neu_prev = var_extr.sel(time=var_extr.time.dt.year.isin(neutral_years - 1))

    neu_cur_mean = var_neu_cur.groupby("time.dayofyear").mean("time")
    neu_cur_std  = var_neu_cur.groupby("time.dayofyear").std("time")
    neu_prev_mean = var_neu_prev.groupby("time.dayofyear").mean("time")
    neu_prev_std  = var_neu_prev.groupby("time.dayofyear").std("time")

    neu_cur_mean = np.array(neu_cur_mean)
    neu_cur_std  = np.array(neu_cur_std)
    neu_prev_mean = np.array(neu_prev_mean)
    neu_prev_std  = np.array(neu_prev_std)

    n_extreme = len(low_years)
    var_low_cur_arr  = np.reshape(np.array(var_low_cur), (n_extreme, n_days))
    var_low_prev_arr = np.reshape(np.array(var_low_prev), (n_extreme, n_days))

    colors_ext = cm.twilight(np.linspace(0, 1, n_extreme))

    for j in range(n_extreme):
        ax.plot(
            range(N_PREV, n_days),
            var_low_cur_arr[j, 0:n_days - N_PREV],
            color=colors_ext[j],
            alpha=0.8,
            linewidth=2,
            label="low O3 years" if j == 0 else None,
        )
        ax.plot(
            range(0, N_PREV),
            var_low_prev_arr[j, n_days - N_PREV:n_days],
            color=colors_ext[j],
            alpha=0.8,
            linewidth=2,
        )

    ax.errorbar(
        range(N_PREV, n_days),
        neu_cur_mean[0:n_days - N_PREV],
        neu_cur_std[0:n_days - N_PREV],
        color="grey",
        alpha=0.5,
        elinewidth=1.5,
        linewidth=3,
        label="all other years",
    )
    ax.errorbar(
        range(0, N_PREV),
        neu_prev_mean[n_days - N_PREV:n_days],
        neu_prev_std[n_days - N_PREV:n_days],
        color="grey",
        alpha=0.5,
        elinewidth=1.5,
        linewidth=3,
    )

    ax.set_xlim(0, n_days)
    ax.set_xticks([0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334])
    ax.set_xticklabels(
        ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr",
         "May", "June", "July", "Aug", "Sep"],
        fontsize=15,
    )

    ax.set_ylabel(f"QRS_TOT over 60–90°N, {level_label} (K/s)", fontsize=18)
    ax.tick_params(axis="y", labelsize=18)

    ax.legend(fontsize=14)
    ax.set_title(
        f"Polar-cap QRS_TOT (60–90°N), {level_label} — low O$_3$ years vs others"
    )

    # ---- AUTO y-axis formatting (no FixedFormatter!) ----
    series_for_axis = [
        var_low_cur_arr, var_low_prev_arr,
        neu_cur_mean, neu_prev_mean,
        neu_cur_mean - neu_cur_std, neu_cur_mean + neu_cur_std,
        neu_prev_mean - neu_prev_std, neu_prev_mean + neu_prev_std,
    ]
    auto_set_yaxis(ax, series_for_axis, level_label, debug_tag=" EXTREMES")

    plt.savefig(out_png, dpi=300)
    plt.savefig(out_pdf)
    plt.close(fig)

    print("[BlockB_QRS] Saved extreme vs other years figure:")
    print("   ", out_png)
    print("   ", out_pdf)


def plot_special_vs_climatology(
    var_extr,
    var_merged,
    lowest25_years,
    level_label,
    out_png,
    out_pdf,
):
    years_extr = np.unique(var_extr.time.dt.year.values.astype(int))
    n_days = int(var_extr.time.dt.dayofyear.max())
    print(f"[BlockB_QRS] EXTR years ({level_label}):", years_extr)
    print(f"[BlockB_QRS] Days per year:", n_days)

    # ===== EXTR all-year 日气候态 =====
    clim_all_mean = var_extr.groupby("time.dayofyear").mean("time")
    clim_all_std  = var_extr.groupby("time.dayofyear").std("time")

    # ===== low-25% 气候态：Y-1 Oct–Dec + Y Jan–Sep =====
    base_low25_cur = var_extr.sel(time=var_extr.time.dt.year.isin(lowest25_years))
    base_low25_prev = var_extr.sel(time=var_extr.time.dt.year.isin(lowest25_years - 1))

    clim_low25_cur_mean = base_low25_cur.groupby("time.dayofyear").mean("time")
    clim_low25_cur_std  = base_low25_cur.groupby("time.dayofyear").std("time")
    clim_low25_prev_mean = base_low25_prev.groupby("time.dayofyear").mean("time")
    clim_low25_prev_std  = base_low25_prev.groupby("time.dayofyear").std("time")

    clim_all_mean = np.array(clim_all_mean)
    clim_all_std  = np.array(clim_all_std)

    clim_low25_cur_mean = np.array(clim_low25_cur_mean)
    clim_low25_cur_std  = np.array(clim_low25_cur_std)
    clim_low25_prev_mean = np.array(clim_low25_prev_mean)
    clim_low25_prev_std  = np.array(clim_low25_prev_std)

    # all-years 组合
    all_prev_mean = clim_all_mean[n_days - N_PREV:n_days]
    all_prev_std  = clim_all_std[n_days - N_PREV:n_days]
    all_cur_mean  = clim_all_mean[0:n_days - N_PREV]
    all_cur_std   = clim_all_std[0:n_days - N_PREV]
    all_comp_mean = np.concatenate([all_prev_mean, all_cur_mean])
    all_comp_std  = np.concatenate([all_prev_std,  all_cur_std])

    # low-25% 组合
    low25_prev_mean = clim_low25_prev_mean[n_days - N_PREV:n_days]
    low25_prev_std  = clim_low25_prev_std[n_days - N_PREV:n_days]
    low25_cur_mean  = clim_low25_cur_mean[0:n_days - N_PREV]
    low25_cur_std   = clim_low25_cur_std[0:n_days - N_PREV]
    low25_comp_mean = np.concatenate([low25_prev_mean, low25_cur_mean])
    low25_comp_std  = np.concatenate([low25_prev_std,  low25_cur_std])

    # ===== merged special years =====
    years_merged = np.unique(var_merged.time.dt.year.values.astype(int))
    print(f"[BlockB_QRS] merged years ({level_label}):", years_merged)

    rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 10,
        "axes.linewidth": 0.8,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })

    fig, ax = plt.subplots(1, 1, figsize=(6.5, 4.0), constrained_layout=True)

    x_full = np.arange(n_days)
    colors_spec = plt.cm.tab10(np.linspace(0, 1, len(YEARS_SPECIAL)))
    handles_years = []

    for i, y in enumerate(YEARS_SPECIAL):
        if y not in years_merged:
            continue

        cur = var_merged.sel(time=var_merged.time.dt.year == y)
        prev = var_merged.sel(time=var_merged.time.dt.year == (y - 1))

        if (cur.size < n_days) or (prev.size < n_days):
            print(f"⚠️ Year {y:04d} or {y-1:04d} incomplete, skip.")
            continue

        cur_arr  = np.array(cur)
        prev_arr = np.array(prev)

        series_comp = np.concatenate([
            prev_arr[n_days - N_PREV:n_days],
            cur_arr[0:n_days - N_PREV],
        ])

        ax.plot(
            x_full, series_comp,
            linestyle="-", linewidth=1.5,
            color=colors_spec[i], label=f"Year {int(y):04d}",
        )
        handles_years.append(
            Line2D([0], [0], color=colors_spec[i], lw=1.5, label=f"Year {int(y):04d}")
        )

    ax.fill_between(
        x_full,
        all_comp_mean - all_comp_std,
        all_comp_mean + all_comp_std,
        color="0.85", alpha=0.9, linewidth=0, zorder=0,
    )
    ax.plot(
        x_full, all_comp_mean,
        linestyle="--", linewidth=1.8, color="black",
        label="EXTR climatology (all years)",
    )

    ax.fill_between(
        x_full,
        low25_comp_mean - low25_comp_std,
        low25_comp_mean + low25_comp_std,
        facecolor="none", edgecolor="0.5",
        hatch="///", linewidth=0.0, zorder=1,
    )
    ax.plot(
        x_full, low25_comp_mean,
        linestyle="-", linewidth=2.0, color="black",
        label="EXTR low-ozone composite",
    )

    ax.set_xlim(0, n_days)
    ax.set_xticks([0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334])
    ax.set_xticklabels(
        ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr",
         "May", "June", "July", "Aug", "Sep"]
    )
    ax.set_ylabel(f"QRS_TOT over 60–90°N, {level_label} (K/s)")

    # ---- AUTO y-axis formatting ----
    series_for_axis = [
        all_comp_mean, all_comp_mean - all_comp_std, all_comp_mean + all_comp_std,
        low25_comp_mean, low25_comp_mean - low25_comp_std, low25_comp_mean + low25_comp_std,
    ]
    auto_set_yaxis(ax, series_for_axis, level_label, debug_tag=" SPECIAL")

    patch_all = Patch(facecolor="0.85", edgecolor="none", label="EXTR all-year ±1σ")
    patch_low = Patch(facecolor="none", edgecolor="0.5", hatch="///",
                      label="EXTR low-ozone ±1σ")
    line_all = Line2D([0], [0], color="black", lw=1.8, ls="--",
                      label="EXTR climatology (all years)")
    line_low = Line2D([0], [0], color="black", lw=2.0, ls="-",
                      label="EXTR low-ozone composite")

    handles = handles_years + [line_all, patch_all, line_low, patch_low]
    ax.legend(handles=handles, loc="best", frameon=False, fontsize=8, ncol=2)

    ax.set_title(f"Polar-cap QRS_TOT (60–90°N), {level_label}")
    ax.grid(False)

    plt.savefig(out_png, dpi=300)
    plt.savefig(out_pdf)
    plt.close(fig)

    print("[BlockB_QRS] Saved special-years vs climatology figure:")
    print("   ", out_png)
    print("   ", out_pdf)


def main_blockB_QRS():
    qrs_extr = xr.load_dataarray(QRS_EXTR_NC)      # (time, lev[Pa])
    qrs_merged = xr.load_dataarray(QRS_MERGED_NC)  # (time, lev[Pa])

    lev_vals = qrs_extr["lev"].values

    lowest10_years = np.loadtxt(LOW10_TXT, dtype=int).reshape(-1)
    lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)
    print("[BlockB_QRS] Lowest 10 O3 years:", lowest10_years)
    print("[BlockB_QRS] Lowest 25% O3 years:", lowest25_years)

    target_levels_hpa = [10.0, 50.0, 100.0]

    for target_hpa in target_levels_hpa:
        idx, lev_actual_hpa = get_level_index(lev_vals, target_hpa)
        level_label = f"{lev_actual_hpa:.1f} hPa"

        var_extr = qrs_extr.isel(lev=idx)
        var_m = qrs_merged.isel(lev=idx)

        print(f"\n[BlockB_QRS] Processing ~{target_hpa} hPa (index {idx}), "
              f"actual {lev_actual_hpa:.2f} hPa")

        out_png1 = os.path.join(
            QRS_ROOT, f"QRS_TOT_60_90N_{int(round(lev_actual_hpa))}hPa_low10_vs_others_EXTR.png"
        )
        out_pdf1 = os.path.join(
            QRS_ROOT, f"QRS_TOT_60_90N_{int(round(lev_actual_hpa))}hPa_low10_vs_others_EXTR.pdf"
        )
        plot_extremes_vs_others(
            var_extr=var_extr,
            lowest10_years=lowest10_years,
            level_label=level_label,
            out_png=out_png1,
            out_pdf=out_pdf1,
        )

        out_png2 = os.path.join(
            QRS_ROOT,
            f"QRS_TOT_60_90N_{int(round(lev_actual_hpa))}hPa_special_vs_EXTRclim.png"
        )
        out_pdf2 = os.path.join(
            QRS_ROOT,
            f"QRS_TOT_60_90N_{int(round(lev_actual_hpa))}hPa_special_vs_EXTRclim.pdf"
        )
        plot_special_vs_climatology(
            var_extr=var_extr,
            var_merged=var_m,
            lowest25_years=lowest25_years,
            level_label=level_label,
            out_png=out_png2,
            out_pdf=out_pdf2,
        )

    print("[BlockB_QRS] Done.")


if __name__ == "__main__":
    main_blockB_QRS()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockC (QRS_TOT) — 60–90°N QRS_TOT 垂直时间剖面 anomaly + 显著性
#
# 输出 NetCDF:
#   QRS_TOT_pcap_vert_anom_special_60_90N.nc
#
# coords:
#   ref_year (n_ref) : [8, 13, 14, 19]
#   lev             : 垂直层 (Pa)
#   time_index      : 0..364 （Oct–Sep 拼接）
#   dayofyear       : [275..365, 1..274]
#
# variables (K/s):
#   anom_all      (ref_year, lev, time_index)
#   clim_all_comp (lev, time_index)
#
# significance mask (True = 显著):
#   t_sig_all, b_sig_all
#
# baseline:
#   only EXTR all-years climatology (no low25 baseline here)
#
# nboot = 5000
# ============================================================

import os
import numpy as np
import xarray as xr
from scipy.stats import t as t_dist

QRS_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_QRS_TOT"
os.makedirs(QRS_ROOT, exist_ok=True)

QRS_EXTR_NC   = os.path.join(QRS_ROOT, "QRS_TOT_pcap_EXTR_60_90N_lev_time.nc")
QRS_MERGED_NC = os.path.join(QRS_ROOT, "QRS_TOT_pcap_MERGED_60_90N_lev_time.nc")

OUT_NC = os.path.join(QRS_ROOT, "QRS_TOT_pcap_vert_anom_special_60_90N.nc")

N_PREV = 91
YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)


def bootstrap_ci(data, nboot=5000, alpha=0.05, rng=None):
    data = np.asarray(data)
    data = data[~np.isnan(data)]
    n = data.size
    if n < 2:
        return np.nan, np.nan

    if rng is None:
        rng = np.random.default_rng()

    means = np.empty(nboot)
    for i in range(nboot):
        samp = rng.choice(data, size=n, replace=True)
        means[i] = np.mean(samp)

    low = np.percentile(means, 100 * alpha / 2.0)
    high = np.percentile(means, 100 * (1 - alpha / 2.0))
    return low, high


def compute_significance_for_baseline(
    base_anom,   # DataArray (time, lev)
    anom_ref,    # ndarray (lev, t_len)
    doy_base,    # ndarray (time,)
    doy_comp,    # ndarray (t_len,)
    alpha=0.05,
    nboot=5000,
):
    """
    对 ref anomaly 的每个(lev, ti)，用基线同 DOY 的样本检验显著性：
      - t-test
      - bootstrap mean CI
    """
    base_vals = base_anom.values  # (time, lev)
    lev_n = base_vals.shape[1]
    t_len = anom_ref.shape[1]

    t_sig = np.zeros((lev_n, t_len), dtype=bool)
    b_sig = np.zeros((lev_n, t_len), dtype=bool)

    rng = np.random.default_rng(2025)

    for ti in range(t_len):
        day = int(doy_comp[ti])
        mask_day = (doy_base == day)
        if not np.any(mask_day):
            continue

        day_samples = base_vals[mask_day, :]  # (n_samples, lev)

        for li in range(lev_n):
            col = day_samples[:, li]
            col = col[~np.isnan(col)]
            if col.size < 2:
                continue

            obs = anom_ref[li, ti]

            se = np.std(col, ddof=1) / np.sqrt(col.size)
            if se == 0:
                continue
            tstat = obs / se
            pval = 2 * (1 - t_dist.cdf(abs(tstat), df=col.size - 1))
            t_sig[li, ti] = (pval < alpha)

            lo, hi = bootstrap_ci(col, nboot=nboot, alpha=alpha, rng=rng)
            if np.isnan(lo) or np.isnan(hi):
                continue
            b_sig[li, ti] = not (lo <= obs <= hi)

    return t_sig, b_sig


def main_blockC_QRS():
    # ---- load cached pcap series ----
    qrs_extr = xr.load_dataarray(QRS_EXTR_NC)      # (time, lev[Pa])
    qrs_m = xr.load_dataarray(QRS_MERGED_NC)       # (time, lev[Pa])

    lev = qrs_extr["lev"].values
    lev_n = lev.size

    n_days = int(qrs_extr.time.dt.dayofyear.max())
    print("[BlockC_QRS] n_days =", n_days)

    doy_extr = qrs_extr.time.dt.dayofyear.values.astype(int)
    years_extr = qrs_extr.time.dt.year.values.astype(int)
    print("[BlockC_QRS] EXTR years:", np.unique(years_extr)[0], "→", np.unique(years_extr)[-1])

    # ---- 1) EXTR daily climatology (all years) ----
    clim_all = qrs_extr.groupby("time.dayofyear").mean("time")  # (doy, lev)

    # ---- 2) baseline anomalies for significance ----
    clim_all_sel_for_extr = clim_all.sel(dayofyear=qrs_extr.time.dt.dayofyear)
    base_anom_all = qrs_extr - clim_all_sel_for_extr  # (time, lev)

    # ---- 3) Oct–Sep composite DOY sequence ----
    doy_comp = np.concatenate([
        np.arange(n_days - N_PREV + 1, n_days + 1, dtype=int),  # Oct–Dec
        np.arange(1, n_days - N_PREV + 1, dtype=int),           # Jan–Sep
    ])
    t_len = doy_comp.size
    print("[BlockC_QRS] Composite DOY:", doy_comp[:5], "...", doy_comp[-5:])

    # ---- 4) composite climatology (lev, time_index) ----
    clim_all_comp = clim_all.sel(dayofyear=doy_comp)
    clim_all_comp_vals = clim_all_comp.transpose("lev", "dayofyear").values  # (lev, t_len)

    # ---- 5) special years anomalies & significance ----
    years_m = qrs_m.time.dt.year.values.astype(int)
    print("[BlockC_QRS] merged years available:", np.unique(years_m))

    n_ref = len(YEARS_SPECIAL)
    anom_all_arr = np.zeros((n_ref, lev_n, t_len))
    t_sig_all = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    b_sig_all = np.zeros((n_ref, lev_n, t_len), dtype=bool)

    for yi, y in enumerate(YEARS_SPECIAL):
        if y not in np.unique(years_m):
            print(f"[BlockC_QRS] ⚠️ Year {y:04d} not found in merged, skip.")
            continue

        ref_cur = qrs_m.sel(time=qrs_m.time.dt.year == y)
        ref_prev = qrs_m.sel(time=qrs_m.time.dt.year == (y - 1))

        if ref_cur.time.size < n_days or ref_prev.time.size < n_days:
            print(f"[BlockC_QRS] ⚠️ Year {y:04d} or {y-1:04d} incomplete ({ref_cur.time.size},{ref_prev.time.size}), skip.")
            continue

        ref_cur_vals = ref_cur.transpose("time", "lev").values  # (365, lev)
        ref_prev_vals = ref_prev.transpose("time", "lev").values

        ref_comp_vals = np.concatenate([
            ref_prev_vals[n_days - N_PREV:n_days, :],  # Oct–Dec of Y-1
            ref_cur_vals[0:n_days - N_PREV, :],        # Jan–Sep of Y
        ], axis=0)  # (t_len, lev)

        ref_comp = ref_comp_vals.T  # (lev, t_len)

        anom_all = ref_comp - clim_all_comp_vals
        anom_all_arr[yi, :, :] = anom_all

        print(f"[BlockC_QRS] Significance: year {y:04d} vs ALL baseline ...")
        t_all, b_all = compute_significance_for_baseline(
            base_anom_all,
            anom_all,
            doy_extr,
            doy_comp,
            alpha=0.05,
            nboot=5000,
        )
        t_sig_all[yi, :, :] = t_all
        b_sig_all[yi, :, :] = b_all

    # ---- 6) save nc ----
    ds_out = xr.Dataset(
        coords={
            "ref_year": ("ref_year", YEARS_SPECIAL),
            "lev": ("lev", lev),
            "time_index": ("time_index", np.arange(t_len)),
            "dayofyear": ("time_index", doy_comp),
        },
        data_vars={
            "anom_all": (("ref_year", "lev", "time_index"), anom_all_arr),
            "clim_all_comp": (("lev", "time_index"), clim_all_comp_vals),
            "t_sig_all": (("ref_year", "lev", "time_index"), t_sig_all),
            "b_sig_all": (("ref_year", "lev", "time_index"), b_sig_all),
        },
    )

    ds_out["anom_all"].attrs["units"] = "K/s"
    ds_out["clim_all_comp"].attrs["units"] = "K/s"
    ds_out["lev"].attrs["units"] = "Pa"

    ds_out.attrs["description"] = (
        "60–90N polar-cap cosine-lat weighted mean QRS_TOT time–height anomalies "
        "for special years relative to EXTR all-years climatology. "
        "Oct–Sep composite is previous-year Oct–Dec + current-year Jan–Sep. "
        "Significance masks: t-test and bootstrap (nboot=5000)."
    )

    ds_out.to_netcdf(OUT_NC)
    print("[BlockC_QRS] Saved:", OUT_NC)
    print("[BlockC_QRS] Done.")


if __name__ == "__main__":
    main_blockC_QRS()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockD (QRS_TOT) — 绘制 60–90°N QRS_TOT 时间×高度 anomaly 图
#
# 使用 BlockC 输出：
#   QRS_TOT_pcap_vert_anom_special_60_90N.nc
#
# 对每个 ref_year（8, 13, 14, 19）生成 2 张图：
#   1) t-test 非显著区打 hatch
#   2) bootstrap 非显著区打 hatch
#
# 总图数：4*2 = 8
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

QRS_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_QRS_TOT"
os.makedirs(QRS_ROOT, exist_ok=True)

IN_NC = os.path.join(QRS_ROOT, "QRS_TOT_pcap_vert_anom_special_60_90N.nc")

XTICKS = [0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334]
XTICK_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr",
                "May", "June", "July", "Aug", "Sep"]

mpl.rc("xtick", direction="out", labelsize=8)
mpl.rc("ytick", direction="out", labelsize=8)
mpl.rcParams["xtick.major.width"] = 0.8
mpl.rcParams["ytick.major.width"] = 0.8
mpl.rc("font", family="sans-serif")
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Liberation Sans"]
mpl.rc("axes", titlesize=11, labelsize=9, linewidth=0.8)
mpl.rc("legend", fontsize=7, frameon=False)
mpl.rc("figure", figsize=(6, 4), dpi=300)
mpl.rc("savefig", bbox="tight", pad_inches=0.1)


def guess_p_2_pa(lev_vals):
    lev_vals = np.asarray(lev_vals)
    maxv = np.nanmax(lev_vals)
    if maxv <= 2000.0:
        return lev_vals * 100.0, "hPa"
    else:
        return lev_vals, "Pa"


def plot_time_height_anom(
    x_idx,
    lev_vals,
    anom_vals,
    clim_vals,
    nonsig_mask,
    ref_year,
    test_label,
    outfile,
):
    fig, ax = plt.subplots()

    x = np.asarray(x_idx)
    p_pa, p_unit = guess_p_2_pa(lev_vals)

    # symmetric color limits
    vlim = np.nanmax(np.abs(anom_vals))
    if np.isfinite(vlim) and vlim > 0:
        vmax = vlim
    else:
        vmax = 1e-6
    levels = np.linspace(-vmax, vmax, 31)

    cf = ax.contourf(
        x, p_pa, anom_vals,
        levels=levels,
        cmap="RdBu_r",
        extend="both",
        antialiased=True,
        alpha=0.88,
    )

    # climatology contours
    clim_mean = np.nanmean(clim_vals)
    clim_std = np.nanstd(clim_vals)
    if np.isfinite(clim_std) and clim_std > 0:
        clim_levels = np.linspace(clim_mean - 1.5 * clim_std,
                                  clim_mean + 1.5 * clim_std, 7)
    else:
        clim_levels = 7

    CS = ax.contour(
        x, p_pa, clim_vals,
        levels=clim_levels,
        colors="k",
        linewidths=0.7,
    )
    ax.clabel(CS, inline=True, fontsize=6, fmt="%.2e")

    # nonsignificant hatch
    if nonsig_mask is not None:
        mask_int = nonsig_mask.astype(int)
        ax.contourf(
            x, p_pa, mask_int,
            levels=[0.5, 1.5],
            colors="none",
            hatches=["///"],
            alpha=0,
        )
        patch = Patch(
            facecolor="white",
            hatch="///",
            edgecolor="black",
            label="Not significant (p ≥ 0.05)",
        )
        ax.legend(handles=[patch], loc="upper right")

    ax.set_yscale("log")
    ax.invert_yaxis()
    ax.set_ylabel(f"Pressure ({p_unit})")

    ax.set_xlim(x[0], x[-1])
    ax.set_xticks(XTICKS)
    ax.set_xticklabels(XTICK_LABELS)

    ax.grid(True, which="major", linestyle="--", linewidth=0.4, alpha=0.5)

    ax.set_title(
        f"QRS_TOT anomaly (K/s), 60–90°N, Year {int(ref_year):04d}\n"
        f"Baseline: EXTR all-years climatology, Mask: non-significant by {test_label}"
    )

    cbar = plt.colorbar(cf, ax=ax, pad=0.02)
    cbar.set_label("QRS_TOT anomaly (K/s)")

    plt.savefig(outfile, dpi=300)
    plt.close(fig)
    print("  Saved:", outfile)


def main_blockD_QRS():
    ds = xr.open_dataset(IN_NC)

    ref_years = ds["ref_year"].values
    lev = ds["lev"].values
    time_index = ds["time_index"].values

    anom_all = ds["anom_all"].values    # (ref, lev, time)
    clim_all = ds["clim_all_comp"].values
    t_sig_all = ds["t_sig_all"].values
    b_sig_all = ds["b_sig_all"].values

    ds.close()

    for yi, y in enumerate(ref_years):
        # t-test mask
        nonsig_t = ~t_sig_all[yi, :, :]
        outfile = os.path.join(
            QRS_ROOT,
            f"QRS_TOT_anom_60_90N_year{int(y):04d}_vsALL_t_nonsig.png"
        )
        plot_time_height_anom(
            time_index,
            lev,
            anom_all[yi, :, :],
            clim_all,
            nonsig_t,
            ref_year=y,
            test_label="t-test",
            outfile=outfile,
        )

        # bootstrap mask
        nonsig_b = ~b_sig_all[yi, :, :]
        outfile = os.path.join(
            QRS_ROOT,
            f"QRS_TOT_anom_60_90N_year{int(y):04d}_vsALL_bootstrap_nonsig.png"
        )
        plot_time_height_anom(
            time_index,
            lev,
            anom_all[yi, :, :],
            clim_all,
            nonsig_b,
            ref_year=y,
            test_label="bootstrap",
            outfile=outfile,
        )

    print("[BlockD_QRS] All 8 figures generated.")


if __name__ == "__main__":
    main_blockD_QRS()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockE (QRS_TOT) — 计算 lat×lev 季节结构（clim + special + anom）
# 【最终版】merged 无条件 interp 到 EXTR lat/lev 网格后再算 anomaly。
#
# 输出 NetCDF:
#   QRS_TOT_latlev_seasonal_EXTRclim_and_special_anom.nc
#
# coords:
#   season   : ["DJF","MAM","JJA","SON"]
#   lev (Pa) : 23 levels
#   lat      : 96
#   ref_year : [8,13,14,19]
#
# data_vars:
#   clim_season(season, lev, lat)              EXTR climatology
#   spec_season(ref_year, season, lev, lat)   special seasonal mean
#   anom_season(ref_year, season, lev, lat)   special - clim
# ============================================================

import os
import numpy as np
import xarray as xr

QRS_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_QRS_TOT"
os.makedirs(QRS_ROOT, exist_ok=True)

EXTR_QRS_PATH = (
    "/mnt/backup_ETH/extr_2000/CO2x1SmidEmin_yBWCN/"
    "CO2x1SmidEmin_yBWCN.cam.h1.0101-0300_T2Mz_QRS_TOT.isobar.nc"
)
MERGED_QRS_PATH = (
    "/mnt/backup_ETH/marina/waccm/CHEM_2000_restart/BWCN.e122.f19_g16.002/"
    "BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc"
)

OUT_NC = os.path.join(
    QRS_ROOT, "QRS_TOT_latlev_seasonal_EXTRclim_and_special_anom.nc"
)

YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)
SEASONS_ORDER = ["DJF", "MAM", "JJA", "SON"]


def lev_to_pa(lev_vals):
    lev_vals = np.asarray(lev_vals)
    if np.nanmax(lev_vals) <= 2000.0:  # hPa
        return lev_vals * 100.0
    return lev_vals


def compare_coords(name, a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    same_len = (a.size == b.size)
    print(f"[COORD-CHECK] {name}: len(a)={a.size}, len(b)={b.size}, same_len={same_len}")
    if same_len:
        max_abs_diff = float(np.nanmax(np.abs(a - b)))
        print(f"[COORD-CHECK] {name}: max|a-b| = {max_abs_diff:.3e}")
        for k in [0, 1, 2, -3, -2, -1]:
            print(f"   a[{k}]={a[k]:.8f}, b[{k}]={b[k]:.8f}, diff={a[k]-b[k]:.3e}")


def build_extr_clim_season():
    ds = xr.open_dataset(EXTR_QRS_PATH)
    da = ds["QRS_TOT"]  # (time, lev[hPa], lat)

    lev_pa = lev_to_pa(ds["lev"].values)
    da = da.assign_coords(lev=("lev", lev_pa))
    da["lev"].attrs["units"] = "Pa"

    clim = da.groupby("time.season").mean("time")  # (season, lev, lat)
    clim = clim.sel(season=SEASONS_ORDER)

    lat_extr = ds["lat"]
    lev_extr = clim["lev"]

    print("=== [BlockE_QRS] EXTR climatology built ===")
    print("clim dims:", clim.dims, "shape:", clim.shape)
    print("lat size:", lat_extr.size, "lev size:", lev_extr.size)
    ds.close()
    return clim, lat_extr, lev_extr


def special_season_means(da_y, da_y1):
    mon_y = da_y.time.dt.month
    mon_y1 = da_y1.time.dt.month

    djf = xr.concat(
        [da_y1.sel(time=mon_y1 == 12),
         da_y.sel(time=(mon_y == 1) | (mon_y == 2))],
        dim="time"
    ).mean("time")

    mam = da_y.sel(time=(mon_y >= 3) & (mon_y <= 5)).mean("time")
    jja = da_y.sel(time=(mon_y >= 6) & (mon_y <= 8)).mean("time")
    son = da_y.sel(time=(mon_y >= 9) & (mon_y <= 11)).mean("time")

    out = xr.concat([djf, mam, jja, son], dim="season")
    out = out.assign_coords(season=("season", SEASONS_ORDER))
    return out  # (season, lev, lat)


def main_blockE_QRS():
    # ---- 1) EXTR seasonal climatology ----
    clim_season, lat_extr, lev_extr = build_extr_clim_season()

    # ---- 2) merged raw zonal mean ----
    ds_m = xr.open_dataset(MERGED_QRS_PATH)
    da_m_raw = ds_m["QRS_TOT"]          # (time, plev, lat, lon)
    da_m = da_m_raw.mean("lon")         # (time, plev, lat)

    lev_pa_m = lev_to_pa(ds_m["plev"].values)
    da_m = da_m.rename({"plev": "lev"}).assign_coords(lev=("lev", lev_pa_m))
    da_m["lev"].attrs["units"] = "Pa"

    print("\n=== [BlockE_QRS] MERGED zonal-mean ready (before interp) ===")
    print("merged dims:", da_m.dims, "shape:", da_m.shape)
    print("merged lat size:", da_m["lat"].size, "lev size:", da_m["lev"].size)

    compare_coords("lat(before)", lat_extr.values, da_m["lat"].values)
    compare_coords("lev(Pa)(before)", lev_extr.values, da_m["lev"].values)

    # ---- 2.1) 强制插值到 EXTR 网格 ----
    print("\n[BlockE_QRS] >>> Forcing interpolation to EXTR lat/lev grid (no checks).")
    da_m = da_m.interp(
        lat=lat_extr,
        lev=lev_extr,
        kwargs={"fill_value": "extrapolate"},
    )
    print("[BlockE_QRS] Interp done.")
    print("merged dims after interp:", da_m.dims, "shape:", da_m.shape)

    compare_coords("lat(after)", lat_extr.values, da_m["lat"].values)
    compare_coords("lev(Pa)(after)", lev_extr.values, da_m["lev"].values)

    years_m = np.unique(da_m.time.dt.year.values.astype(int))
    print("\nmerged years span:", years_m[0], "→", years_m[-1])

    # ---- 3) special seasonal means & anomalies ----
    spec_list, anom_list, used_years = [], [], []

    for y in YEARS_SPECIAL:
        if (y not in years_m) or ((y - 1) not in years_m):
            print(f"[BlockE_QRS] ⚠️ missing year {y:04d} or {y-1:04d}, skip")
            continue

        da_y  = da_m.sel(time=da_m.time.dt.year == y)
        da_y1 = da_m.sel(time=da_m.time.dt.year == (y - 1))

        print(f"\n[BlockE_QRS] Year {y:04d}: da_y size={da_y.time.size}, da_y1 size={da_y1.time.size}")

        spec_season = special_season_means(da_y, da_y1)  # (season, lev, lat)
        anom_season = spec_season - clim_season          # now lat must stay 96

        print("[BlockE_QRS] spec_season:", spec_season.shape, spec_season.dims)
        print("[BlockE_QRS] clim_season:", clim_season.shape, clim_season.dims)
        print("[BlockE_QRS] anom_season:", anom_season.shape, anom_season.dims)

        spec_list.append(spec_season)
        anom_list.append(anom_season)
        used_years.append(y)

    used_years = np.array(used_years, dtype=int)

    spec_all = xr.concat(spec_list, dim="ref_year").assign_coords(ref_year=used_years)
    anom_all = xr.concat(anom_list, dim="ref_year").assign_coords(ref_year=used_years)

    print("\n=== [BlockE_QRS] FINAL arrays ===")
    print("clim_season:", clim_season.shape)
    print("spec_all   :", spec_all.shape)
    print("anom_all   :", anom_all.shape)

    # ---- 4) save nc ----
    ds_out = xr.Dataset(
        coords={
            "season": ("season", SEASONS_ORDER),
            "ref_year": ("ref_year", used_years),
            "lev": ("lev", lev_extr.values),
            "lat": ("lat", lat_extr.values),
        },
        data_vars={
            "clim_season": (("season", "lev", "lat"), clim_season.values),
            "spec_season": (("ref_year", "season", "lev", "lat"), spec_all.values),
            "anom_season": (("ref_year", "season", "lev", "lat"), anom_all.values),
        },
    )

    for v in ["clim_season", "spec_season", "anom_season"]:
        ds_out[v].attrs["units"] = "K/s"
    ds_out["lev"].attrs["units"] = "Pa"

    ds_out.attrs["description"] = (
        "Seasonal lat–lev structures of QRS_TOT. "
        "clim_season: EXTR all-years climatology (DJF/MAM/JJA/SON). "
        "spec_season: special-year seasonal means with DJF=Dec(Y-1)+Jan/Feb(Y). "
        "anom_season = spec_season - clim_season. "
        "Merged data forced-interpolated to EXTR lat/lev grid before anomaly."
    )

    ds_out.to_netcdf(OUT_NC)
    print("\n[BlockE_QRS] Saved seasonal lat-lev cache to:", OUT_NC)

    ds_m.close()
    print("[BlockE_QRS] Done.")


if __name__ == "__main__":
    main_blockE_QRS()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockF (QRS_TOT) — 读 BlockE 输出 nc 并绘制 lat×lev 2×2 组图
#
# 图：
#   1) EXTR climatology四季 2×2 (1张)
#   2) 每个 special year anomaly四季 2×2 (4张)
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt

QRS_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_QRS_TOT"
IN_NC = os.path.join(QRS_ROOT, "QRS_TOT_latlev_seasonal_EXTRclim_and_special_anom.nc")

SEASONS_ORDER = ["DJF", "MAM", "JJA", "SON"]


def plot_2x2_latlev(field_season, lat, lev_pa, title_prefix, out_png, out_pdf,
                    cmap="viridis", symmetric=False):
    mpl.rc("xtick", direction="out", labelsize=8)
    mpl.rc("ytick", direction="out", labelsize=8)
    mpl.rc("font", family="sans-serif")
    mpl.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Liberation Sans"]
    mpl.rc("axes", titlesize=10, labelsize=9, linewidth=0.8)

    p_pa = lev_pa

    vals = field_season.values
    if symmetric:
        vlim = float(np.nanpercentile(np.abs(vals), 98))
        vmin, vmax = -vlim, vlim
        levels = np.linspace(vmin, vmax, 31)
    else:
        vmin = float(np.nanpercentile(vals, 2))
        vmax = float(np.nanpercentile(vals, 98))
        levels = np.linspace(vmin, vmax, 31)

    fig, axes = plt.subplots(2, 2, figsize=(8.2, 6.2), constrained_layout=True)
    axes = axes.ravel()

    for i, s in enumerate(SEASONS_ORDER):
        ax = axes[i]
        field = field_season.sel(season=s).transpose("lev", "lat").values  # (lev, lat)

        cf = ax.contourf(lat, p_pa, field, levels=levels, cmap=cmap, extend="both")

        ax.set_yscale("log")
        ax.invert_yaxis()
        ax.set_title(f"{title_prefix} {s}")
        ax.set_xlabel("Latitude (deg)")
        ax.set_ylabel("Pressure (hPa)")

        # nice pressure ticks
        yticks_pa = np.array([1000, 500, 200, 100, 50, 20, 10, 5, 1, 0.1]) * 100.0
        ax.set_yticks(yticks_pa)
        ax.set_yticklabels([1000, 500, 200, 100, 50, 20, 10, 5, 1, 0.1])

    cbar = fig.colorbar(cf, ax=axes, orientation="vertical", pad=0.02, shrink=0.9)
    cbar.set_label("QRS_TOT anomaly (K/s)" if symmetric else "QRS_TOT (K/s)")

    plt.savefig(out_png, dpi=300)
    plt.savefig(out_pdf)
    plt.close(fig)
    print("  Saved:", out_png)
    print("  Saved:", out_pdf)


def main_blockF_QRS():
    ds = xr.open_dataset(IN_NC)

    clim = ds["clim_season"]          # (season, lev, lat)
    anom = ds["anom_season"]          # (ref_year, season, lev, lat)
    ref_years = ds["ref_year"].values
    lat = ds["lat"].values
    lev = ds["lev"].values

    print("=== [BlockF_QRS] Read cache ===")
    print("clim shape:", clim.shape, "anom shape:", anom.shape)
    print("lat:", lat.size, "lev:", lev.size, "ref_years:", ref_years)

    # ---- 1) plot climatology ----
    out_png = os.path.join(QRS_ROOT, "QRS_TOT_lat-lev_climatology_4seasons_EXTR.png")
    out_pdf = os.path.join(QRS_ROOT, "QRS_TOT_lat-lev_climatology_4seasons_EXTR.pdf")
    plot_2x2_latlev(
        clim, lat, lev,
        title_prefix="EXTR climatology",
        out_png=out_png, out_pdf=out_pdf,
        cmap="viridis", symmetric=False
    )

    # ---- 2) plot anomalies per special year ----
    for yi, y in enumerate(ref_years):
        field = anom.sel(ref_year=y)

        out_png = os.path.join(
            QRS_ROOT, f"QRS_TOT_lat-lev_anom_4seasons_year{int(y):04d}_vsEXTRclim.png"
        )
        out_pdf = os.path.join(
            QRS_ROOT, f"QRS_TOT_lat-lev_anom_4seasons_year{int(y):04d}_vsEXTRclim.pdf"
        )

        plot_2x2_latlev(
            field, lat, lev,
            title_prefix=f"Year {int(y):04d} anomaly",
            out_png=out_png, out_pdf=out_pdf,
            cmap="RdBu_r", symmetric=True
        )

    ds.close()
    print("[BlockF_QRS] Done.")


if __name__ == "__main__":
    main_blockF_QRS()
